In [1]:
import sys

sys.path.append('../../../')

In [2]:
from __future__ import annotations

import copy
import warnings
from typing import Union, Optional, Sequence

import torch
import torch.nn as nn
import torch.utils.checkpoint as cp
from torch.nn.modules.utils import _ntuple, _triple

%load_ext autoreload
%autoreload 2

from computer_vision.slowfast.mmcv.cnn.bricks.conv_module import ConvModule
from computer_vision.slowfast.mmcv.cnn.brick import build_activation_layer
from computer_vision.slowfast.mmaction.models.backbones.resnet3d import BasicBlock3d, Bottleneck3d, ResNet3d

In [3]:
fast_pathway={
  'depth': 50,
  'pretrained': None,
  'base_channels': 8,
  'conv1_kernel': (5, 7, 7),
  'conv1_stride_t': 1,
  'pool1_stride_t': 1,
  'norm_eval': False}
resnet=ResNet3d(**fast_pathway, verbose=True)
x=torch.rand(3,3,10,180,180)
print(f"{x.shape=}, ({x.min().item()}{x.max().item()})")
outs=resnet(x)
print(f'{type(outs)=}, {outs.shape=}, ({outs.min().item()}, {outs.max().item()})')
nn.MSELoss()(outs, torch.rand_like(outs)).backward()

stage 0 --------------------
spatial_stride=1, temporal_stride=1, temporal_stride=1, self.inplanes=8, planes=8
downsample module: (spatial_stride!=1)=False, (inplanes!=planes*block.expansion)=True
resblock 0: inplanes=8, planes=8 (inflate[0]==1)=True, (non_local[0]==1)=False, inflate_style='3x1x1'
resblock 1: inplanes=32, planes=8 (inflate[0]==1)=True, (non_local[0]==1)=False, inflate_style='3x1x1'
resblock 2: inplanes=32, planes=8 (inflate[0]==1)=True, (non_local[0]==1)=False, inflate_style='3x1x1'
stage 1 --------------------
spatial_stride=2, temporal_stride=1, temporal_stride=1, self.inplanes=32, planes=16
downsample module: (spatial_stride!=1)=True, (inplanes!=planes*block.expansion)=True
resblock 0: inplanes=32, planes=16 (inflate[0]==1)=True, (non_local[0]==1)=False, inflate_style='3x1x1'
resblock 1: inplanes=64, planes=16 (inflate[0]==1)=True, (non_local[0]==1)=False, inflate_style='3x1x1'
resblock 2: inplanes=64, planes=16 (inflate[0]==1)=True, (non_local[0]==1)=False, inflate

In [4]:
fast_pathway={
  'depth': 50,
  'pretrained': None,
  'base_channels': 8,
  'conv1_kernel': (5, 7, 7),
  'conv1_stride_t': 1,
  'pool1_stride_t': 1,
  'norm_eval': True}
resnet=ResNet3d(**fast_pathway, verbose=True)
resnet.train()
x=torch.rand(3,3,10,180,180)
print(f"{x.shape=}, ({x.min().item()}{x.max().item()})")
outs=resnet(x)
print(f'{type(outs)=}, {outs.shape=}, ({outs.min().item()}, {outs.max().item()})')
nn.MSELoss()(outs, torch.rand_like(outs)).backward()

stage 0 --------------------
spatial_stride=1, temporal_stride=1, temporal_stride=1, self.inplanes=8, planes=8
downsample module: (spatial_stride!=1)=False, (inplanes!=planes*block.expansion)=True
resblock 0: inplanes=8, planes=8 (inflate[0]==1)=True, (non_local[0]==1)=False, inflate_style='3x1x1'
resblock 1: inplanes=32, planes=8 (inflate[0]==1)=True, (non_local[0]==1)=False, inflate_style='3x1x1'
resblock 2: inplanes=32, planes=8 (inflate[0]==1)=True, (non_local[0]==1)=False, inflate_style='3x1x1'
stage 1 --------------------
spatial_stride=2, temporal_stride=1, temporal_stride=1, self.inplanes=32, planes=16
downsample module: (spatial_stride!=1)=True, (inplanes!=planes*block.expansion)=True
resblock 0: inplanes=32, planes=16 (inflate[0]==1)=True, (non_local[0]==1)=False, inflate_style='3x1x1'
resblock 1: inplanes=64, planes=16 (inflate[0]==1)=True, (non_local[0]==1)=False, inflate_style='3x1x1'
resblock 2: inplanes=64, planes=16 (inflate[0]==1)=True, (non_local[0]==1)=False, inflate